# Challenge Lab 5

Deploy the Lab 4 agent to Agent Engine and test it.


In [1]:
!pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk]>=1.112" google-adk


## Setup


In [2]:
import os
import vertexai

project_id = !gcloud config get-value project
PROJECT_ID = project_id[0].strip()
LOCATION = "us-central1"
STAGING_BUCKET = f"gs://{PROJECT_ID}-bucket"
DISPLAY_NAME = "challenge-lab-5-research-pipeline"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

!gcloud services enable aiplatform.googleapis.com storage.googleapis.com
!gcloud storage buckets describe {STAGING_BUCKET} >/dev/null 2>&1 || gcloud storage buckets create {STAGING_BUCKET} --project={PROJECT_ID} --location={LOCATION} --uniform-bucket-level-access

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)
print(PROJECT_ID)
print(LOCATION)
print(STAGING_BUCKET)

Operation "operations/acat.p2-659576290083-5aed0cc2-c93e-4587-994a-dbcba4c05912" finished successfully.
qwiklabs-gcp-00-93a45da1459f
us-central1
gs://qwiklabs-gcp-00-93a45da1459f-bucket


In [3]:
!mkdir -p research_pipeline_agent_engine


In [4]:
%%writefile research_pipeline_agent_engine/__init__.py


Overwriting research_pipeline_agent_engine/__init__.py


In [5]:
%%writefile research_pipeline_agent_engine/agent.py
from google.adk.agents import LlmAgent, SequentialAgent, LoopAgent, ParallelAgent
from google.adk.tools import google_search
from google.adk.tools.tool_context import ToolContext
from google.adk.models.google_llm import Gemini
from google.genai import types

MODEL = Gemini(
    model="gemini-2.5-flash",
    retry_options=types.HttpRetryOptions(initial_delay=1, attempts=3),
)


def exit_loop(tool_context: ToolContext):
    tool_context.actions.escalate = True
    return {"status": "loop_exited"}


primary_researcher = LlmAgent(
    name="primary_researcher",
    model=MODEL,
    description="Searches for direct answers and supporting evidence.",
    instruction=(
        "Search for factual information to answer the user's question. "
        "Use multiple searches if needed. Cite sources.\n\n"
        "Keep it to bullet points and key facts."
    ),
    tools=[google_search],
    output_key="primary_research",
)

counter_researcher = LlmAgent(
    name="counter_researcher",
    model=MODEL,
    description="Finds counterpoints and weak spots.",
    instruction=(
        "Find criticisms, counterarguments, missing context, and inconvenient facts.\n\n"
        "Do not write a balanced conclusion. Report only the pushback in bullet points."
    ),
    tools=[google_search],
    output_key="counter_research",
)

research_team = ParallelAgent(
    name="research_team",
    sub_agents=[primary_researcher, counter_researcher],
)


synthesizer = LlmAgent(
    name="synthesizer",
    model=MODEL,
    description="Builds the draft answer.",
    include_contents="none",
    instruction=(
        "MAIN FINDINGS:\n{primary_research}\n\n"
        "COUNTERPOINTS:\n{counter_research}\n\n"
        "Write one clear answer that uses both. Keep it under 400 words."
    ),
    output_key="draft_answer",
)


critic = LlmAgent(
    name="critic",
    model=MODEL,
    description="Checks the draft.",
    include_contents="none",
    instruction=(
        "Review this draft answer:\n\n"
        "DRAFT: {draft_answer}\n\n"
        "If it is solid, respond with exactly APPROVED. Otherwise list the specific issues."
    ),
    output_key="critique",
)

refiner = LlmAgent(
    name="refiner",
    model=MODEL,
    description="Fixes the draft or exits the loop.",
    include_contents="none",
    instruction=(
        "DRAFT: {draft_answer}\n\n"
        "CRITIQUE: {critique}\n\n"
        "If the critique says APPROVED, call exit_loop and do not output text. "
        "Otherwise rewrite the draft to fix every issue."
    ),
    tools=[exit_loop],
    output_key="draft_answer",
)

review_loop = LoopAgent(
    name="review_loop",
    sub_agents=[critic, refiner],
    max_iterations=2,
)


answer_pipeline = SequentialAgent(
    name="answer_pipeline",
    sub_agents=[research_team, synthesizer, review_loop],
)


root_agent = LlmAgent(
    name="greeter",
    model=MODEL,
    description="Entry point for the pipeline.",
    instruction=(
        "You are a research assistant. When the user asks a question, delegate to answer_pipeline."
    ),
    sub_agents=[answer_pipeline],
)


Overwriting research_pipeline_agent_engine/agent.py


## Local test


In [6]:
import importlib.util
from vertexai.agent_engines import AdkApp

spec = importlib.util.spec_from_file_location(
    "research_pipeline_agent_engine",
    "research_pipeline_agent_engine/agent.py",
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

app = AdkApp(agent=mod.root_agent, app_name="research_pipeline_agent_engine")
print("Local app ready")


async def run_local(query, user_id="challenge-lab-5-user"):
    async for event in app.async_stream_query(user_id=user_id, message=query):
        print(event)


Local app ready


In [7]:
await run_local("What caused the 2024 CrowdStrike outage?")


{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-c36d0648-c912-4a2c-92b9-82aa0b844304', 'args': {'agent_name': 'answer_pipeline'}, 'name': 'transfer_to_agent'}, 'thought_signature': 'CtYCAY89a19YjTvQ6BWx8b4833CBBupm_ir2F74BY42L564K_dGQ2Xt5wOynug8UJzNPD28MCeXfkA0ahkFQAz_roChg_HLNmf6ZnKdNq1W-p2soYkc8Cbu4h6psJkvDyAPa4MXxP-aFH0aocAkKADijvM0dUcjiWKudKlRRxdVL7iFqiwHlWCJnwbkvvp7dyPYGmNfGa35YfKJp3s4T9nm0RtvCczbq_r4j26wOU7h2G6ijy_6QhrjctXbRTKOfl2qExCSIfM1pykJ-MvJJ_Q0IuXWSDdv2IgZJMfpShrvbAHDDhUInO4SpqUdXVvy3tlMCDsaiFWqQj08e4vAgpAZKtohueEQXp9xu8mUZ__OVelJrRAvoV0BkfV2u2olG1XGsRDc01Vj8aZwlotZ9OKnGkasD85rO3KmO0sUhDmjJoAI1s2vy5LmkKNIcLe582fJWshmkiqPt'}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 11, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 11}], 'prompt_token_count': 299, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 299}], 'thoughts_token_count': 65, 'total_token_count': 37

## Deploy


In [8]:
from vertexai import agent_engines
import importlib.util

spec = importlib.util.spec_from_file_location(
    "research_pipeline_agent_engine",
    "research_pipeline_agent_engine/agent.py",
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

remote_agent = agent_engines.create(
    agent_engine=mod.root_agent,
    requirements=[
        "google-cloud-aiplatform[agent_engines,adk]>=1.112",
        "google-adk",
    ],
    display_name=DISPLAY_NAME,
    extra_packages=["./research_pipeline_agent_engine"],
)

print(remote_agent.resource_name)

INFO:vertexai.agent_engines:Deploying google.adk.agents.Agent as an application.
INFO:vertexai.agent_engines:Identified the following requirements: {'google-cloud-aiplatform': '1.140.0', 'pydantic': '2.12.3', 'cloudpickle': '3.1.2'}
INFO:vertexai.agent_engines:The following requirements are appended: {'pydantic==2.12.3', 'cloudpickle==3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]>=1.112', 'google-adk', 'pydantic==2.12.3', 'cloudpickle==3.1.2']
INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-00-93a45da1459f-bucket
INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-00-93a45da1459f-bucket/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-00-93a45da1459f-bucket/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-00-93a45da1459f-bucket/agent_engine/dependencies.tar.gz
INFO

projects/659576290083/locations/us-central1/reasoningEngines/6514147898224017408


## Remote test


In [9]:
async def run_remote(query, user_id="challenge-lab-5-user"):
    session = await remote_agent.async_create_session(user_id=user_id)
    session_id = session.id if hasattr(session, "id") else session["id"]
    print(session_id)

    async for event in remote_agent.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        message=query,
    ):
        print(event)


In [10]:
await run_remote("What caused the 2024 CrowdStrike outage?")


1125188110502592512
{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-b3bb6b2f-3053-4683-a55a-2dda29f49224', 'args': {'agent_name': 'answer_pipeline'}, 'name': 'transfer_to_agent'}, 'thought_signature': 'CtQDAY89a19Q6OhSgYEozjWxvmQH0hjeLElkcHHnb574bZIlc2B2j8JZMAMS0YirHwjhMax3WUHGs_Sfqil680mfHAkY0Rp2F3-3yu5MHnbhe8VQubNfn-lorZc1m_Zr0G8iOeu49L2oq973djgI7AY2Ym__bZx9LLzz9_SK352mOK14w6S4CsOzqvk5Rc1DxbuaHtWgpvaC_OqePvRWC90Ys20l_d7VyIlpNaoCfx6-dLhJV7k4JWS1-H9ZN7Q2cq92PNmXSKnYG2SjkTHAHWpHncKmL86Tv20TefEwKN6l1SZUAdVpUk1mzwr-IiTumUMdSgm71FTmvEUwzsUbu5p4MEOeQUZm5xtax17nOtWsObSWT00kCfKRzdngm_yattjLGxzIwiR7obJTBfRnbBK41_fVySPxVg3osgaWqYo-GCcBoHMOM1j75HcS2Dkn3VfMIoGa-SFPo-hx1fp1DlHPAJXCBUYq6o0aHZJ8u8yI35ZoXnNnUDEqxTX7faM1qzLMNuJuovDZSPOqKHYkdtBzDEf_uLjYA9c74uLsBwTljWjlxo73yaVziiNEnoSqUKSc2giemqf4zsj1Bzast929WmoqDHwCAAaG6SfPPpg92ASoTIId'}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 11, 'candidates_tokens_details': [{